# When Accuracy Saturates — staged accuracy and reliability experiments

This notebook runs the staged accuracy/reliability study of 3D segmentation on MSD Task07 (pancreas) and MSD Task04 (hippocampus): Stage 1 compares architectures for accuracy; Stage 2 compares uncertainty estimators for reliability on the same held-out test set.

**What it runs**
1. **Test-set audit (pancreas).** Rebuilds the fixed held-out test set from the cache and checks that it equals the reference case list in `data/pancreas_test_cases.txt`, and that no test case appears in any seed's training/validation pool.
2. **Stage 1 (pancreas), extended.** Five-seed runs of the baseline 3D U-Net, the Fresnel variant, the spectral edge-attention (SEA) module, and a dropout variant of the baseline (needed for MC Dropout). Paired Wilcoxon tests with Holm correction over the whole family, per-seed signs, two one-sided equivalence tests (TOST) with prespecified margins, and the minimum detectable difference at 80% power.
3. **Stage 2 (pancreas), on the Stage-1 test set.** Entropy, test-time augmentation, deep ensemble, MC Dropout, Fresnel feature coherence, and Fresnel decision-space disagreement. Every estimator is scored with its *own* prediction and confidence; metrics are computed in the boundary band, per class, and over the whole volume; per-volume run time is recorded.
4. **Statistics.** Patient-level bootstrap 95% confidence intervals; paired Wilcoxon tests against entropy with Holm correction over the full family (estimators × metrics); the patient is the unit of analysis.
5. **Qualitative panels.** CT, ground truth, prediction, error and uncertainty maps for representative test cases.
6. **Hippocampus.** Stages 1 and 2 on MSD Task04 with the same protocol.

**Runtime.** Several hours on a Colab GPU. Every training run is checkpointed to Drive; if the runtime disconnects, re-run the notebook from the top and finished work is skipped. Every training run is checkpointed and the notebook is resumable: re-running a cell skips work already done. All long loops print progress. Training stops early when validation Dice has not improved by more than `min_delta` for `early_stop_patience` epochs.

**Storage.** The MSD Task07 archive is downloaded to the Colab disk (`/content`) and the 112³ cache is built there. The compact cache (uint8 .npz) is copied to `MyDrive/Datasets/PancreasCache_112/`, so a restarted runtime reuses it instead of downloading the archive again. Clone the repository to `/content` first so the reference test list is found. Checkpoints (≈1.4 MB each), tables, figures and `results.json` are written to `MyDrive/Outputs/Accuracy-Saturation/`.


In [ ]:
# Cell 1 — Environment (Google Colab)
from google.colab import drive
drive.mount('/content/drive')
import sys, subprocess, importlib
for pkg in ["nibabel"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.run([sys.executable,"-m","pip","install","-q",pkg],check=True)
import os, json, time, math, shutil, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch
warnings.filterwarnings("ignore")
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# Cell 2 — Configuration
from dataclasses import dataclass
DRIVE = Path("/content/drive/MyDrive")
SESSION_BUDGET_H = 11.5   # stop starting new training runs before the Colab runtime limit; re-run to resume
SESSION_T0 = time.time()
def time_left_ok(margin_min=10):
    return (time.time()-SESSION_T0)/3600 < SESSION_BUDGET_H - margin_min/60

@dataclass
class Config:
    # ---- inputs (read only) ----
    pancreas_cache: Path = DRIVE/"Datasets"/"PancreasCache_112"                    # 112^3 uint8 cache
    test_list:     Path = Path("/content/Accuracy_Saturation_Reliability_3DSeg/data/pancreas_test_cases.txt")  # fixed held-out pancreas test set
    # ---- outputs ----
    out: Path = DRIVE/"Outputs"/"Accuracy-Saturation"
    local: Path = Path("/content/work")
    # ---- protocol ----
    target_shape: tuple = (112,112,112)
    test_seed: int = 12345
    test_fraction: float = 0.15
    val_fraction: float = 0.15
    seeds: tuple = (0,1,2,3,4)
    base_channels: int = 16
    n_bands: int = 4
    n_dist: int = 3
    epochs: int = 40
    batch_size: int = 1
    lr: float = 2e-3
    weight_decay: float = 1e-5
    w_dice: float = 1.0; w_ce: float = 1.0
    amp: bool = True
    early_stop_patience: int = 8       # stop after this many epochs without a significant gain
    min_delta: float = 0.003           # a validation-Dice gain below 0.003 is not significant
    min_epochs: int = 10               # early stopping cannot trigger before this epoch
    fp32_arches: tuple = ()            # architectures to train in float32 with gradient clipping
    clip_norm: float = 1.0
    # ---- new arms ----
    new_arches: tuple = ("sea","dropout")
    dropout_p: float = 0.1             # Dropout3d in the decoder of the dropout variant
    # ---- Stage 2 ----
    tta_K: int = 6                     # identity + 5 augmented passes (Gaussian noise sd 0.03 + axis flips)
    tta_noise_sd: float = 0.03
    mc_T: int = 10                     # MC Dropout stochastic passes
    band_dilation: int = 3             # evaluation band = dilation of (pred ∪ GT) foreground
    ece_bins: int = 10
    n_boot: int = 2000
    # ---- equivalence margins (absolute Dice), fixed before analysis ----
    eq_margin: tuple = (("pancreas",0.03),("mass",0.05),("anterior",0.02),("posterior",0.02))
    # ---- hippocampus ----
    run_hippocampus: bool = True
    hippo_shape: tuple = (48,64,48)
    hippo_epochs: int = 60
    hippo_url: str = "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task04_Hippocampus.tar"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
cfg = Config()
for d in [cfg.out, cfg.out/"Checkpoints", cfg.out/"Tables", cfg.out/"Figures", cfg.local]:
    d.mkdir(parents=True, exist_ok=True)
LOG = cfg.out/"run_log.txt"
def log(msg):
    line=f"[{time.strftime('%H:%M:%S')}] {msg}"; print(line, flush=True)
    with open(LOG,"a") as f: f.write(line+"\n")
log(f"config ready | outputs -> {cfg.out}")


In [ ]:
# Cell 3 — Models + dropout variant
import math, torch.nn as nn

class MultiBandSpectralEdgeAttention3D(nn.Module):
    def __init__(self, channels, n_bands=4):
        super().__init__()
        self.mu=nn.Parameter(torch.linspace(0.45,0.90,n_bands).clone())
        self.log_sigma=nn.Parameter(torch.log(torch.full((n_bands,),0.08)))
        self.fuse=nn.Conv3d(channels*n_bands,channels,1)
        self.gate=nn.Sequential(nn.Conv3d(channels,channels,1),nn.Sigmoid()); self.n=n_bands
    def _rho(s,D,H,W,dev):
        fz=torch.fft.fftfreq(D,device=dev);fy=torch.fft.fftfreq(H,device=dev);fx=torch.fft.rfftfreq(W,device=dev)
        Z,Y,X=torch.meshgrid(fz,fy,fx,indexing="ij"); r=torch.sqrt(Z**2+Y**2+X**2); return r/(r.max()+1e-6)
    def forward(s,x):
        D,H,W=x.shape[-3:]; Xf=torch.fft.rfftn(x.float(),dim=(-3,-2,-1)); rho=s._rho(D,H,W,x.device)
        sg=torch.exp(s.log_sigma); mu=torch.clamp(s.mu,0,1)
        bands=[torch.fft.irfftn(Xf*torch.exp(-((rho-mu[b])**2)/(2*sg[b]**2+1e-6)).to(Xf.dtype),s=(D,H,W),dim=(-3,-2,-1)).to(x.dtype) for b in range(s.n)]
        return x*(1.0+s.gate(s.fuse(torch.cat(bands,1))))

class FresnelPropagation3D(nn.Module):
    def __init__(self, channels, n_dist=3):
        super().__init__()
        self.z=nn.Parameter(torch.linspace(0.05,0.30,n_dist)); self.fuse=nn.Conv3d(channels*n_dist,channels,1); self.n=n_dist; self.capture=False; self.last=None
    def _k2(s,D,H,W,dev):
        fz=torch.fft.fftfreq(D,device=dev);fy=torch.fft.fftfreq(H,device=dev);fx=torch.fft.rfftfreq(W,device=dev)
        Z,Y,X=torch.meshgrid(fz,fy,fx,indexing="ij"); return Z**2+Y**2+X**2
    def forward(s,x):
        D,H,W=x.shape[-3:]; Xf=torch.fft.rfftn(x.float(),dim=(-3,-2,-1)); k2=s._k2(D,H,W,x.device)
        outs=[torch.fft.irfftn(Xf*torch.exp(1j*(2*math.pi)*s.z[i]*k2).to(Xf.dtype),s=(D,H,W),dim=(-3,-2,-1)).to(x.dtype) for i in range(s.n)]
        if getattr(s,'capture',False): s.last=torch.stack(outs,0).detach()
        return x+s.fuse(torch.cat(outs,1))

class ResBlock3D(nn.Module):
    def __init__(s,ci,co):
        super().__init__(); s.c1=nn.Conv3d(ci,co,3,padding=1); s.n1=nn.InstanceNorm3d(co)
        s.c2=nn.Conv3d(co,co,3,padding=1); s.n2=nn.InstanceNorm3d(co); s.skip=nn.Conv3d(ci,co,1) if ci!=co else nn.Identity()
    def forward(s,x): h=torch.relu(s.n1(s.c1(x))); h=s.n2(s.c2(h)); return torch.relu(h+s.skip(x))

def make_module(arch, ch, cfg):
    if arch=="sea": return MultiBandSpectralEdgeAttention3D(ch, cfg.n_bands)
    if arch=="fresnel": return FresnelPropagation3D(ch, cfg.n_dist)
    return nn.Identity()

class CompareUNet3D(nn.Module):
    def __init__(s, cfg, n_classes, arch="baseline", in_ch=1):
        super().__init__(); base=cfg.base_channels
        s.e1=ResBlock3D(in_ch,base); s.e2=ResBlock3D(base,2*base); s.bott=ResBlock3D(2*base,4*base)
        s.up2=nn.ConvTranspose3d(4*base,2*base,2,2); s.d2=ResBlock3D(4*base,2*base)
        s.up1=nn.ConvTranspose3d(2*base,base,2,2); s.d1=ResBlock3D(2*base,base)
        s.m2=make_module(arch,2*base,cfg); s.m1=make_module(arch,base,cfg)
        s.out=nn.Conv3d(base,n_classes,1); s.pool=nn.MaxPool3d(2)
    def forward(s,x):
        e1=s.e1(x); e2=s.e2(s.pool(e1)); b=s.bott(s.pool(e2))
        d2=s.m2(s.d2(torch.cat([s.up2(b),e2],1))); d1=s.m1(s.d1(torch.cat([s.up1(d2),e1],1)))
        return s.out(d1)




class DropResBlock3D(ResBlock3D):
    """Residual block with Dropout3d after the first convolution (decoder only)."""
    def __init__(s,ci,co,p): super().__init__(ci,co); s.drop=nn.Dropout3d(p)
    def forward(s,x): h=torch.relu(s.n1(s.c1(x))); h=s.drop(h); h=s.n2(s.c2(h)); return torch.relu(h+s.skip(x))

class DropoutUNet3D(CompareUNet3D):
    """Baseline backbone with dropout in both decoder blocks; used for MC Dropout."""
    def __init__(s,cfg,n_classes,in_ch=1):
        super().__init__(cfg,n_classes,"baseline",in_ch); base=cfg.base_channels
        s.d2=DropResBlock3D(4*base,2*base,cfg.dropout_p); s.d1=DropResBlock3D(2*base,base,cfg.dropout_p)

def build_model(arch, cfg, n_classes):
    if arch=="dropout": return DropoutUNet3D(cfg,n_classes)
    return CompareUNet3D(cfg,n_classes,arch)
log("models ready")


In [ ]:
# Cell 4 — Losses and metrics
import torch.nn.functional as F
from scipy.ndimage import distance_transform_edt, binary_erosion, gaussian_filter

def dice_loss_mc(logits,t,n,eps=1.):
    p=torch.softmax(logits,1); oh=F.one_hot(t,n).permute(0,4,1,2,3).float()
    num=2*(p[:,1:]*oh[:,1:]).sum((2,3,4))+eps; den=p[:,1:].sum((2,3,4))+oh[:,1:].sum((2,3,4))+eps
    return (1-num/den).mean()
def total_loss(logits,t,cfg,n): return cfg.w_dice*dice_loss_mc(logits,t,n)+cfg.w_ce*F.cross_entropy(logits,t)
def dice_bin(p,t,eps=1e-6): p=p.astype(bool);t=t.astype(bool); return float((2*(p&t).sum()+eps)/(p.sum()+t.sum()+eps))
def hd95_bin(p,t):
    p=p.astype(bool);t=t.astype(bool)
    if p.sum()==0 or t.sum()==0: return float("nan")
    dt_t=distance_transform_edt(~t);dt_p=distance_transform_edt(~p)
    sp=p&~binary_erosion(p,border_value=0); st=t&~binary_erosion(t,border_value=0)
    d=np.concatenate([dt_t[sp],dt_p[st]]); return float(np.percentile(d,95)) if d.size else float("nan")
def per_class(pred,t,n): return {c:(dice_bin(pred==c,t==c),hd95_bin(pred==c,t==c)) for c in range(1,n)}
def meanfg(pred,t,n): return float(np.mean([dice_bin(pred==c,t==c) for c in range(1,n)]))

log("losses and metrics ready")


In [ ]:
# Cell 5 — Pancreas data and test-set audit
import torch.nn.functional as F
import nibabel as nib, tarfile, re
LOCAL_CACHE = cfg.local/"PancreasCache"
HU = (-100.0, 240.0)   # abdominal window
if list(cfg.pancreas_cache.glob("*.npz")):
    if not LOCAL_CACHE.exists():
        log("copying pancreas cache to the Colab disk ..."); shutil.copytree(cfg.pancreas_cache, LOCAL_CACHE)
else:
    # No cache supplied: rebuild it from the raw MSD Task07 archive with the study's preprocessing
    # (HU window -100..240 -> 0..255, trilinear resize of the image, nearest resize of the labels, 112^3).
    RAW = cfg.local/"Task07_Pancreas"
    if not (RAW/"imagesTr").exists():
        FULL = 12289971712   # size of the complete MSD archive; partial uploads are ignored
        tars = [p for p in list(cfg.pancreas_cache.glob("Task07_Pancreas*.tar")) + list(cfg.local.glob("Task07_Pancreas*.tar")) if p.stat().st_size == FULL]
        if not tars:
            tarp = cfg.local/"Task07_Pancreas.tar"; tarp.parent.mkdir(parents=True, exist_ok=True)
            log("downloading MSD Task07 Pancreas (12.3 GB) to the Colab disk ...")
            import urllib.request
            with urllib.request.urlopen("https://msd-for-monai.s3-us-west-2.amazonaws.com/Task07_Pancreas.tar") as r, open(tarp, "wb") as f:
                got = 0
                while True:
                    b = r.read(64 << 20)
                    if not b: break
                    f.write(b); got += len(b)
                    if got % (1 << 30) < (64 << 20): log(f"   {got/1e9:.1f} / 12.3 GB")
            assert tarp.stat().st_size == FULL, "incomplete download, re-run this cell"
            tars=[tarp]
        log(f"extracting {tars[0].name} to the Colab disk ...")
        with tarfile.open(tars[0]) as t: t.extractall(cfg.local)
    LOCAL_CACHE.mkdir(parents=True, exist_ok=True)
    imgs = sorted(p for p in (RAW/"imagesTr").glob("*.nii.gz") if not p.name.startswith("."))
    lo, hi = HU
    for k, ip in enumerate(imgs):
        cid = ip.name[:-7].lower(); out = LOCAL_CACHE/f"{cid}.npz"
        if out.exists(): continue
        img = np.asarray(nib.load(str(ip)).get_fdata(), np.float32)
        lab = np.round(np.asarray(nib.load(str(RAW/"labelsTr"/ip.name)).get_fdata())).astype(np.int64)
        img = ((np.clip(img, lo, hi)-lo)/(hi-lo)*255.0)
        img = F.interpolate(torch.from_numpy(img)[None,None].float(), size=cfg.target_shape, mode="trilinear", align_corners=False)[0,0].numpy().clip(0,255).astype(np.uint8)
        lab = F.interpolate(torch.from_numpy(lab.astype(np.float32))[None,None], size=cfg.target_shape, mode="nearest")[0,0].numpy().astype(np.uint8)
        np.savez_compressed(out, img=img, lab=lab)
        if (k+1) % 20 == 0: log(f"   cached {k+1}/{len(imgs)}")
    log(f"pancreas cache rebuilt: {len(list(LOCAL_CACHE.glob('*.npz')))} volumes")
    # keep a copy of the compact cache in Drive so a restarted runtime skips the 12 GB download
    cfg.pancreas_cache.mkdir(parents=True, exist_ok=True)
    for p in sorted(LOCAL_CACHE.glob("*.npz")): shutil.copy(p, cfg.pancreas_cache/p.name)
    mb=sum(p.stat().st_size for p in cfg.pancreas_cache.glob("*.npz"))/1e6
    log(f"cache copied to {cfg.pancreas_cache} ({mb:.0f} MB); raw archive removed from the Colab disk")
    shutil.rmtree(cfg.local/"Task07_Pancreas", ignore_errors=True)
    for t in cfg.local.glob("Task07_Pancreas*.tar"): t.unlink()
cache_files = sorted(p for p in LOCAL_CACHE.glob("*.npz"))
log(f"pancreas cache: {len(cache_files)} volumes")

def detect_classes(files, sample=60):
    vals=set()
    for p in files[:sample]: vals.update(int(v) for v in np.unique(np.load(p)["lab"]) if v>0)
    fg=sorted(vals); return {v:i+1 for i,v in enumerate(fg)}, len(fg)
LMAP, NFG = detect_classes(cache_files); NCLS = NFG+1
CLASS_NAMES = {1:"pancreas",2:"mass"} if NFG==2 else {c:f"class{c}" for c in range(1,NCLS)}

def split_from(files, n_files=None):
    """Original split rule: rng(test_seed) permutation over the sorted cache list."""
    allidx=np.arange(len(files)); np.random.default_rng(cfg.test_seed).shuffle(allidx)
    ntest=int(len(files)*cfg.test_fraction)
    return [files[i] for i in allidx[:ntest]], [files[i] for i in allidx[ntest:]]

test_files, trainval_files = split_from(cache_files)
test_ids = {p.stem for p in test_files}

# Reference test set: the case list shipped with the repository (data/pancreas_test_cases.txt).
# The rebuilt split must reproduce it exactly.
ref=cfg.test_list
if ref.exists():
    stage1_ids=set(ref.read_text().split())
    same=test_ids==stage1_ids
    log(f"rebuilt test set: {len(test_ids)} cases | reference list: {len(stage1_ids)} cases | identical: {same}")
    if not same:
        test_files=[p for p in cache_files if p.stem in stage1_ids]
        trainval_files=[p for p in cache_files if p.stem not in stage1_ids]
        assert len(test_files)==len(stage1_ids), "some reference test cases are missing from the cache"
else:
    stage1_ids=test_ids; same=True; log(f"rebuilt test set: {len(test_ids)} cases (no reference list found)")

def split_trainval(seed):
    ix=np.arange(len(trainval_files)); np.random.default_rng(seed).shuffle(ix)
    nv=max(1,int(len(trainval_files)*cfg.val_fraction))
    return [trainval_files[i] for i in ix[nv:]], [trainval_files[i] for i in ix[:nv]]

def load_npz(p, lmap=LMAP):
    d=np.load(p); img=d["img"].astype(np.float32); raw=d["lab"].astype(np.int64)
    img=(img-img.mean())/(img.std()+1e-6)
    lab=np.zeros_like(raw)
    for v,k in lmap.items(): lab[raw==v]=k
    return img.astype(np.float32), lab

class DS(torch.utils.data.Dataset):
    def __init__(s,files,lmap,aug=False): s.f=files; s.lmap=lmap; s.aug=aug
    def __len__(s): return len(s.f)
    def __getitem__(s,i):
        img,lab=load_npz(s.f[i],s.lmap)
        if s.aug and np.random.rand()<0.5: img=img+np.random.normal(0,0.05,img.shape).astype(np.float32)
        return torch.from_numpy(img)[None], torch.from_numpy(lab), s.f[i].stem


In [ ]:
# Cell 6 — Training (min-delta early stopping), resumable
from torch.utils.data import DataLoader

def train_one(arch, tr_files, va_files, ckpt_path, seed, ncls, lmap, epochs):
    torch.manual_seed(seed); np.random.seed(seed)
    net=build_model(arch,cfg,ncls).to(cfg.device)
    opt=torch.optim.Adam(net.parameters(),cfg.lr,weight_decay=cfg.weight_decay)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,epochs)
    stable=arch in cfg.fp32_arches                       # full precision + gradient clipping for numerically fragile models
    use_amp=cfg.amp and cfg.device=="cuda" and not stable
    scaler=torch.amp.GradScaler('cuda',enabled=use_amp)
    tr=DataLoader(DS(tr_files,lmap,True),batch_size=cfg.batch_size,shuffle=True,num_workers=2)
    va=DataLoader(DS(va_files,lmap,False),batch_size=1,num_workers=2)
    best=-1; ref=-1; bs=None; since=0; t0=time.time()
    for ep in range(epochs):
        net.train(); ls=[]
        for x,y,_ in tr:
            x,y=x.to(cfg.device),y.to(cfg.device); opt.zero_grad()
            with torch.amp.autocast('cuda',enabled=use_amp):
                loss=total_loss(net(x),y,cfg,ncls)
            scaler.scale(loss).backward()
            if stable: scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(net.parameters(), cfg.clip_norm)
            scaler.step(opt); scaler.update(); ls.append(loss.item())
        sch.step(); net.eval(); vd=[]
        with torch.no_grad():
            for x,y,_ in va: vd.append(meanfg(net(x.to(cfg.device)).argmax(1).cpu().numpy()[0],y.numpy()[0],ncls))
        vd=float(np.mean(vd))
        if vd>best: bs={k:v.detach().cpu().clone() for k,v in net.state_dict().items()}   # keep the best weights
        if vd>ref+cfg.min_delta: ref=vd; since=0                                      # significant gain resets the counter
        else: since+=1
        best=max(best,vd)
        log(f"    {arch} seed{seed} ep {ep+1}/{epochs} loss {np.mean(ls):.4f} val {vd:.4f} best {best:.4f} "
            f"({time.time()-t0:.0f}s)")
        if ep+1>=cfg.min_epochs and since>=cfg.early_stop_patience: log(f"    early stop at epoch {ep+1} (no gain > {cfg.min_delta} in val Dice for {since} epochs)"); break
    net.load_state_dict(bs); torch.save(bs,ckpt_path); return net

@torch.no_grad()
def eval_dice(net, arch, seed, files, ncls, lmap, names, out_csv):
    net.eval(); rows=[]
    for p in files:
        img,lab=load_npz(p,lmap); pr=net(torch.from_numpy(img)[None,None].to(cfg.device)).argmax(1).cpu().numpy()[0]
        for c,(dc,hd) in per_class(pr,lab,ncls).items():
            rows.append(dict(arch=arch,seed=seed,case_id=p.stem,klass=c,klass_name=names[c],dice=dc,hd95=hd))
    pd.DataFrame(rows).to_csv(out_csv,index=False)

def ckpt_for(task, arch, seed):
    return cfg.out/"Checkpoints"/f"{task}_{arch}_seed{seed}.pt"


def run_stage1(task, arches, files_test, splitter, ncls, lmap, names, epochs):
    for seed in cfg.seeds:
        tr,va=splitter(seed)
        for arch in arches:
            ck=ckpt_for(task,arch,seed); csv=cfg.out/"Tables"/f"{task}_per_case_{arch}_seed{seed}.csv"
            if ck.exists() and csv.exists(): log(f"  {task} {arch} seed{seed}: done, skipped"); continue
            if not ck.exists() and not time_left_ok(45):
                log("  Session time budget reached — restart the notebook in a new session to continue."); return False
            if ck.exists():
                net=build_model(arch,cfg,ncls).to(cfg.device); net.load_state_dict(torch.load(ck,map_location=cfg.device))
                log(f"  {task} {arch} seed{seed}: checkpoint found, evaluating")
            else:
                log(f"  {task} {arch} seed{seed}: training (train={len(tr)} val={len(va)})")
                net=train_one(arch,tr,va,ck,seed,ncls,lmap,epochs)
            eval_dice(net,arch,seed,files_test,ncls,lmap,names,csv); del net; torch.cuda.empty_cache()

REUSE_STAGE1=False

PANC_ARCHES=("baseline","fresnel")+cfg.new_arches
log(f"pancreas Stage 1: arches={PANC_ARCHES} seeds={cfg.seeds}")
run_stage1("pancreas",PANC_ARCHES,test_files,split_trainval,NCLS,LMAP,CLASS_NAMES,cfg.epochs)
log("pancreas Stage 1 training complete")


In [ ]:
# Cell 7 — Stage 1 statistics: Wilcoxon + Holm, per-seed signs, TOST equivalence, power
from scipy import stats
def holm(p):
    p=np.asarray(p,float); o=np.argsort(p); m=len(p); adj=np.empty(m); run=0
    for r,i in enumerate(o): run=max(run,min(1,(m-r)*p[i])); adj[i]=run
    return adj
def safe_wilcoxon(a,b):
    try: return float(stats.wilcoxon(a,b).pvalue)
    except ValueError: return 1.0   # all paired differences zero
def tost(d, margin):
    """Paired TOST on per-case differences; returns max of the two one-sided p-values and the 90% CI."""
    n=len(d); m=d.mean(); se=d.std(ddof=1)/np.sqrt(n)
    p_lo=1-stats.t.cdf((m+margin)/se,n-1); p_hi=stats.t.cdf((m-margin)/se,n-1)
    ci=stats.t.interval(0.90,n-1,loc=m,scale=se); return max(p_lo,p_hi), ci
def mdd(d, alpha=0.05, power=0.80):
    """Minimum detectable paired mean difference (two-sided, normal approximation)."""
    return (stats.norm.ppf(1-alpha/2)+stats.norm.ppf(power))*d.std(ddof=1)/np.sqrt(len(d))

def stage1_stats(task, arches):
    df=pd.concat([pd.read_csv(f) for f in sorted((cfg.out/"Tables").glob(f"{task}_per_case_*_seed*.csv"))])
    margins=dict(cfg.eq_margin); rows=[]; sign=[]
    for kl in sorted(df.klass_name.unique()):
        base=df[(df.arch=="baseline")&(df.klass_name==kl)].groupby("case_id").dice.mean()
        for a in [x for x in arches if x!="baseline"]:
            arm=df[(df.arch==a)&(df.klass_name==kl)].groupby("case_id").dice.mean()
            c=base.index.intersection(arm.index); d=(arm[c]-base[c]).values
            p=safe_wilcoxon(arm[c],base[c]); pt,ci=tost(d,margins.get(kl,0.03))
            rows.append(dict(task=task,klass=kl,arch=a,n=len(c),base_mean=base[c].mean(),arm_mean=arm[c].mean(),
                             delta=d.mean(),ci90_low=ci[0],ci90_high=ci[1],p_wilcoxon=p,margin=margins.get(kl,0.03),
                             p_tost=pt,mdd80=mdd(d)))
            for s in cfg.seeds:
                b=df[(df.arch=="baseline")&(df.seed==s)&(df.klass_name==kl)].set_index("case_id").dice
                r=df[(df.arch==a)&(df.seed==s)&(df.klass_name==kl)].set_index("case_id").dice
                cc=b.index.intersection(r.index)
                if len(cc): sign.append(dict(task=task,klass=kl,arch=a,seed=s,delta=(r[cc]-b[cc]).mean()))
    res=pd.DataFrame(rows); res["p_holm"]=holm(res.p_wilcoxon); res["p_tost_holm"]=holm(res.p_tost)
    res["equivalent"]=res.p_tost_holm<0.05; sign=pd.DataFrame(sign)
    summ=df.groupby(["arch","klass_name"]).apply(lambda g: pd.Series(dict(
        dice_mean=g.dice.mean(), seed_std=g.groupby("seed").dice.mean().std(), n_seeds=g.seed.nunique()))).reset_index()
    res.to_csv(cfg.out/"Tables"/f"{task}_stage1_tests.csv",index=False)
    sign.to_csv(cfg.out/"Tables"/f"{task}_stage1_per_seed_delta.csv",index=False)
    summ.to_csv(cfg.out/"Tables"/f"{task}_stage1_summary.csv",index=False)
    log(f"{task} Stage 1 summary:\n"+summ.round(4).to_string(index=False))
    log(f"{task} Stage 1 tests (Holm over {len(res)} comparisons):\n"+
        res[["klass","arch","delta","ci90_low","ci90_high","p_wilcoxon","p_holm","margin","p_tost_holm","equivalent","mdd80"]].round(4).to_string(index=False))
    return res
panc_s1=stage1_stats("pancreas",PANC_ARCHES)


In [ ]:
# Cell 8 — Stage 2 estimators and metrics
from scipy.ndimage import binary_dilation
from sklearn.metrics import roc_auc_score

def softmax_t(lo): return torch.softmax(lo,1)
@torch.no_grad()
def prob_of(net,img):
    return softmax_t(net(torch.from_numpy(img)[None,None].to(cfg.device)))[0].cpu().numpy()
def entropy_np(p): return -(p*np.log(p+1e-9)).sum(0)
def sync():
    if cfg.device=="cuda": torch.cuda.synchronize()

@torch.no_grad()
def tta(net,img):
    rng=np.random.default_rng(0); sm=[prob_of(net,img)]
    for k in range(1,cfg.tta_K):                     # k=1..5: flips by bit pattern of k + Gaussian noise
        a=img+rng.normal(0,cfg.tta_noise_sd,img.shape).astype(np.float32)
        flips=[ax for ax in (0,1,2) if (k>>ax)&1]
        for ax in flips: a=np.flip(a,ax)
        s=prob_of(net,np.ascontiguousarray(a))
        for ax in flips: s=np.flip(s,ax+1)
        sm.append(np.ascontiguousarray(s))
    return np.mean(sm,0)

@torch.no_grad()
def mc_dropout(net,img):
    net.eval()
    for m in net.modules():
        if isinstance(m,nn.Dropout3d): m.train()     # dropout active, normalisation in eval mode
    p=np.mean([prob_of(net,img) for _ in range(cfg.mc_T)],0)
    net.eval(); return p

@torch.no_grad()
def fresnel_views(net,img):
    """(a) feature-space coherence: variance across propagated views at the last Fresnel layer;
       (b) decision-space disagreement: each view routed separately through the trained fuse + head (mutual information)."""
    for m in net.modules():
        if isinstance(m,FresnelPropagation3D): m.capture=True; m.last=None
    x=torch.from_numpy(img)[None,None].to(cfg.device); p=softmax_t(net(x))[0].cpu().numpy()
    last=[m.last for m in net.modules() if isinstance(m,FresnelPropagation3D) and m.last is not None][-1]
    coh=last.float().var(0).mean(1)[0][None,None]
    coh=F.interpolate(coh,size=img.shape,mode="trilinear",align_corners=False)[0,0].cpu().numpy()
    for m in net.modules():
        if isinstance(m,FresnelPropagation3D): m.capture=False; m.last=None
    s=net; e1=s.e1(x); e2=s.e2(s.pool(e1)); b=s.bott(s.pool(e2))
    d2=s.m2(s.d2(torch.cat([s.up2(b),e2],1))); d1p=s.d1(torch.cat([s.up1(d2),e1],1)); fr=s.m1
    D,H,W=d1p.shape[-3:]; Xf=torch.fft.rfftn(d1p.float(),dim=(-3,-2,-1)); k2=fr._k2(D,H,W,x.device)
    props=[torch.fft.irfftn(Xf*torch.exp(1j*(2*math.pi)*fr.z[i]*k2).to(Xf.dtype),s=(D,H,W),dim=(-3,-2,-1)).to(d1p.dtype) for i in range(fr.n)]
    views=[]
    for i in range(fr.n):
        mk=[torch.zeros_like(props[j]) for j in range(fr.n)]; mk[i]=props[i]
        views.append(softmax_t(s.out(d1p+fr.fuse(torch.cat(mk,1))))[0].cpu().numpy())
    views=np.stack(views,0); mi=np.clip(entropy_np(views.mean(0))-np.mean([entropy_np(v) for v in views],0),0,None)
    return p, coh, mi

def ece(conf,correct,bins):
    e=0.0; n=len(conf)
    for i in range(bins):
        m=(conf>i/bins)&(conf<=(i+1)/bins)
        if m.any(): e+=abs(correct[m].mean()-conf[m].mean())*m.sum()/n
    return float(e)
def rc_auc(conf,correct,steps=20):
    c=correct[np.argsort(-conf,kind="mergesort")]
    return float(np.mean([1-c[:max(1,int(len(c)*q))].mean() for q in np.linspace(0.05,1.0,steps)]))
def score(u,prob,gt,mask):
    """Failure labels: voxel misclassified by this estimator's own argmax prediction."""
    pred=prob.argmax(0); conf=prob.max(0); err=(pred!=gt)[mask]; uu=u[mask]
    au=float(roc_auc_score(err,uu)) if 0<err.sum()<len(err) else np.nan
    corr=(~err).astype(float)
    return dict(auroc=au,ece=ece(conf[mask],corr,cfg.ece_bins),rcauc=rc_auc(conf[mask],corr),
                error_rate=float(err.mean()),n_vox=int(mask.sum()))
def regions(prob,gt,ncls):
    pred=prob.argmax(0)
    R={"band":binary_dilation((pred>0)|(gt>0),iterations=cfg.band_dilation),"volume":np.ones_like(gt,bool)}
    for c in range(1,ncls): R[f"band_c{c}"]=binary_dilation((pred==c)|(gt==c),iterations=cfg.band_dilation)
    return R
log("Stage 2 estimators ready")


In [ ]:
# Cell 9 — Stage 2 run (resumable per task/assessed-architecture), with timing
def load_net(task,arch,seed,ncls):
    net=build_model(arch,cfg,ncls).to(cfg.device)
    net.load_state_dict(torch.load(ckpt_for(task,arch,seed),map_location=cfg.device)); net.eval(); return net

def timed(f,*a):
    sync(); t=time.time(); out=f(*a); sync(); return out, time.time()-t

def run_stage2(task, assess, files_test, ncls, lmap):
    out_csv=cfg.out/"Tables"/f"{task}_stage2_{assess}_per_case.csv"
    done=set(pd.read_csv(out_csv).case.unique()) if out_csv.exists() else set()
    ens=[load_net(task,assess,s,ncls) for s in cfg.seeds]; m0=ens[0]
    mc=load_net(task,"dropout",cfg.seeds[0],ncls) if (cfg.out/"Checkpoints"/f"{task}_dropout_seed{cfg.seeds[0]}.pt").exists() else None
    for i,p in enumerate(files_test):
        if p.stem in done: continue
        img,gt=load_npz(p,lmap); rows=[]
        p0,t1=timed(prob_of,m0,img)
        est={"entropy":(entropy_np(p0),p0,t1)}
        pt,tt=timed(tta,m0,img); est["tta"]=(entropy_np(pt),pt,tt)
        def ensemble(): return np.mean([prob_of(n,img) for n in ens],0)
        pe,te=timed(ensemble); est["ensemble"]=(entropy_np(pe),pe,te)
        if mc is not None:
            pm,tm=timed(mc_dropout,mc,img)
            est["mc_dropout"]=(entropy_np(pm),pm,tm)
        if assess=="fresnel":
            (pf,coh,mi),tf=timed(fresnel_views,m0,img)
            est["fresnel_coh"]=(coh,p0,tf); est["fresnel_disagree"]=(mi,p0,tf)
        for name,(u,prob,sec) in est.items():
            for rname,mask in regions(prob,gt,ncls).items():
                rows.append(dict(task=task,assess=assess,case=p.stem,method=name,region=rname,seconds=sec,**score(u,prob,gt,mask)))
        pd.DataFrame(rows).to_csv(out_csv,mode="a",header=not out_csv.exists(),index=False)
        log(f"  {task}/{assess} stage 2: {i+1}/{len(files_test)} {p.stem}")
    return pd.read_csv(out_csv)

log("pancreas Stage 2 ...")
panc_s2=pd.concat([run_stage2("pancreas",a,test_files,NCLS,LMAP) for a in ("baseline","fresnel")])
log("pancreas Stage 2 complete")


In [ ]:
# Cell 10 — Stage 2 statistics: patient-level bootstrap CIs, Wilcoxon vs entropy, Holm over the family
def boot_ci(x, n=cfg.n_boot, seed=0):
    x=np.asarray(x,float); x=x[~np.isnan(x)]; r=np.random.default_rng(seed)
    b=[r.choice(x,len(x)).mean() for _ in range(n)]; return float(np.percentile(b,2.5)), float(np.percentile(b,97.5))

def stage2_stats(df, task):
    summ=[]; tests=[]
    for (assess,region,method),g in df.groupby(["assess","region","method"]):
        row=dict(task=task,assess=assess,region=region,method=method,n_cases=g.case.nunique(),
                 seconds_per_volume=g.seconds.mean())
        for m in ("auroc","ece","rcauc"):
            row[m]=g[m].mean(); row[m+"_ci_low"],row[m+"_ci_high"]=boot_ci(g[m])
        summ.append(row)
    summ=pd.DataFrame(summ)
    for (assess,region),g in df.groupby(["assess","region"]):
        for method in g.method.unique():
            if method=="entropy": continue
            # Fresnel estimators share the single model's prediction, so only AUROC differs from entropy
            for m in (("auroc",) if method.startswith("fresnel") else ("auroc","ece","rcauc")):
                pv=g.pivot_table(index="case",columns="method",values=m,dropna=False)
                if not {"entropy",method}<=set(pv.columns): continue
                pv=pv[["entropy",method]].dropna()
                if len(pv)<5: continue
                d=pv[method]-pv["entropy"]; better=(d>0) if m=="auroc" else (d<0)
                tests.append(dict(task=task,assess=assess,region=region,method=method,metric=m,n=len(pv),
                                  delta=d.mean(),wins=int(better.sum()),p=safe_wilcoxon(pv[method],pv["entropy"])))
    tests=pd.DataFrame(tests,columns=["task","assess","region","method","metric","n","delta","wins","p"])
    # Holm family: all estimators x metrics vs entropy, within each (task, assessed model, region)
    tests["p_holm"]=np.nan
    if len(tests):
        for k,idx in tests.groupby(["assess","region"]).groups.items(): tests.loc[idx,"p_holm"]=holm(tests.loc[idx,"p"])
    summ.to_csv(cfg.out/"Tables"/f"{task}_stage2_summary.csv",index=False)
    tests.to_csv(cfg.out/"Tables"/f"{task}_stage2_tests.csv",index=False)
    show=summ[summ.region.isin(["band","volume"])][["assess","region","method","auroc","auroc_ci_low","auroc_ci_high","ece","rcauc","seconds_per_volume"]]
    log(f"{task} Stage 2 summary:\n"+show.round(4).to_string(index=False))
    log(f"{task} Stage 2 tests vs entropy (band):\n"+tests[tests.region=="band"].round(5).to_string(index=False))
    return summ,tests
panc_sum,panc_tests=stage2_stats(panc_s2,"pancreas")


In [ ]:
# Cell 11 — Qualitative panels (representative cases: lowest, median and highest ensemble AUROC)
import matplotlib.pyplot as plt
def qualitative(task, assess, files_test, ncls, lmap, tag):
    s2=pd.read_csv(cfg.out/"Tables"/f"{task}_stage2_{assess}_per_case.csv")
    g=s2[(s2.method=="ensemble")&(s2.region=="band")].sort_values("auroc")
    picks=[g.case.iloc[0], g.case.iloc[len(g)//2], g.case.iloc[-1]]
    ens=[load_net(task,assess,s,ncls) for s in cfg.seeds]; m0=ens[0]
    mc=load_net(task,"dropout",cfg.seeds[0],ncls) if (cfg.out/"Checkpoints"/f"{task}_dropout_seed{cfg.seeds[0]}.pt").exists() else None
    byname={p.stem:p for p in files_test}
    cols=["CT","Ground truth","Prediction","Error","Entropy","Ensemble","MC Dropout"]+(["Fresnel disagreement"] if assess=="fresnel" else [])
    fig,ax=plt.subplots(len(picks),len(cols),figsize=(2.3*len(cols),2.4*len(picks)))
    for r,cid in enumerate(picks):
        img,gt=load_npz(byname[cid],lmap); p0=prob_of(m0,img); pred=p0.argmax(0)
        z=int(np.argmax(((gt>0)|(pred>0)).sum((1,2))))
        maps=[img[z],gt[z],pred[z],(pred!=gt)[z].astype(float),entropy_np(p0)[z],
              entropy_np(np.mean([prob_of(n,img) for n in ens],0))[z],
              entropy_np(mc_dropout(mc,img))[z] if mc is not None else np.zeros_like(img[z])]
        if assess=="fresnel": maps.append(fresnel_views(m0,img)[2][z])
        for c,(m,title) in enumerate(zip(maps,cols)):
            a=ax[r,c]; a.axis("off")
            if c in (1,2): a.imshow(img[z],cmap="gray"); a.imshow(np.ma.masked_equal(m,0),cmap="autumn",vmin=0,vmax=ncls-1,alpha=0.6)
            elif c==3: a.imshow(img[z],cmap="gray"); a.imshow(np.ma.masked_equal(m,0),cmap="cool",alpha=0.8)
            elif c>=4: a.imshow(m,cmap="magma")
            else: a.imshow(m,cmap="gray")
            if r==0: a.set_title(title,fontsize=9)
        ax[r,0].text(-5,img.shape[1]//2,cid,rotation=90,va="center",ha="right",fontsize=8)
    plt.tight_layout()
    for ext in ("png","pdf"): fig.savefig(cfg.out/"Figures"/f"{tag}.{ext}",dpi=200,bbox_inches="tight")
    plt.show(); log(f"saved {tag}")
qualitative("pancreas","baseline",test_files,NCLS,LMAP,"pancreas_qualitative_baseline")
qualitative("pancreas","fresnel",test_files,NCLS,LMAP,"pancreas_qualitative_fresnel")


In [ ]:
# Cell 12 — Hippocampus (MSD Task04): cache, Stage 1, Stage 2
import nibabel as nib, tarfile, urllib.request
if cfg.run_hippocampus:
    HROOT=cfg.local/"Task04_Hippocampus"; HCACHE=cfg.local/"HippoCache"
    if not HROOT.exists():
        tarp=cfg.local/"Task04_Hippocampus.tar"
        if not tarp.exists():
            log("downloading MSD Task04 Hippocampus ..."); urllib.request.urlretrieve(cfg.hippo_url,tarp)
        with tarfile.open(tarp) as t: t.extractall(cfg.local)
    HCACHE.mkdir(exist_ok=True)
    imgs=sorted(p for p in (HROOT/"imagesTr").glob("*.nii.gz") if not p.name.startswith("."))
    for k,ip in enumerate(imgs):
        out=HCACHE/(ip.name.replace(".nii.gz",".npz"))
        if out.exists(): continue
        im=np.asarray(nib.load(str(ip)).get_fdata(),np.float32); lb=np.asarray(nib.load(str(HROOT/"labelsTr"/ip.name)).get_fdata())
        lo,hi=np.percentile(im,[0.5,99.5]); im=np.clip((im-lo)/(hi-lo+1e-6),0,1)*255
        t=torch.from_numpy(im)[None,None]; im=F.interpolate(t,size=cfg.hippo_shape,mode="trilinear",align_corners=False)[0,0].numpy()
        l=torch.from_numpy(lb.astype(np.float32))[None,None]; lb=F.interpolate(l,size=cfg.hippo_shape,mode="nearest")[0,0].numpy()
        np.savez_compressed(out,img=im.clip(0,255).astype(np.uint8),lab=lb.astype(np.uint8))
        if (k+1)%50==0: log(f"  hippocampus cache {k+1}/{len(imgs)}")
    hfiles=sorted(HCACHE.glob("*.npz")); log(f"hippocampus cache: {len(hfiles)} volumes")
    HLMAP,HNFG=detect_classes(hfiles); HNCLS=HNFG+1; HNAMES={1:"anterior",2:"posterior"}
    allidx=np.arange(len(hfiles)); np.random.default_rng(cfg.test_seed).shuffle(allidx)
    nt=int(len(hfiles)*cfg.test_fraction); h_test=[hfiles[i] for i in allidx[:nt]]; h_tv=[hfiles[i] for i in allidx[nt:]]
    def h_split(seed):
        ix=np.arange(len(h_tv)); np.random.default_rng(seed).shuffle(ix); nv=max(1,int(len(h_tv)*cfg.val_fraction))
        return [h_tv[i] for i in ix[nv:]],[h_tv[i] for i in ix[:nv]]
    log(f"hippocampus: test={len(h_test)} train/val pool={len(h_tv)}")
    H_ARCHES=("baseline","fresnel","sea","dropout")
    run_stage1("hippocampus",H_ARCHES,h_test,h_split,HNCLS,HLMAP,HNAMES,cfg.hippo_epochs)
    hip_s1=stage1_stats("hippocampus",H_ARCHES)
    hip_s2=pd.concat([run_stage2("hippocampus",a,h_test,HNCLS,HLMAP) for a in ("baseline","fresnel")])
    hip_sum,hip_tests=stage2_stats(hip_s2,"hippocampus")
    qualitative("hippocampus","baseline",h_test,HNCLS,HLMAP,"hippocampus_qualitative_baseline")


In [ ]:
# Cell 13 — Summary bundle (small files only, saved to Drive)
bundle=dict(
  test_set_identical_to_stage1=bool(same), stage1_checkpoints_reused=REUSE_STAGE1,
  config={k:(str(v) if isinstance(v,Path) else v) for k,v in cfg.__dict__.items()},
  pancreas_stage1=pd.read_csv(cfg.out/"Tables"/"pancreas_stage1_tests.csv").to_dict("records"),
  pancreas_stage2=pd.read_csv(cfg.out/"Tables"/"pancreas_stage2_summary.csv").to_dict("records"))
if cfg.run_hippocampus:
    bundle["hippocampus_stage1"]=pd.read_csv(cfg.out/"Tables"/"hippocampus_stage1_tests.csv").to_dict("records")
    bundle["hippocampus_stage2"]=pd.read_csv(cfg.out/"Tables"/"hippocampus_stage2_summary.csv").to_dict("records")
json.dump(bundle,open(cfg.out/"results.json","w"),indent=1,default=float)
log(f"saved {cfg.out/'results.json'}")
